1. Exploratory Data Analysis - FRAUD DATA: FEATURE ENGINEERING (Step 4) FOR FRAUD DATASET ONLY

In [13]:
"""
MINIMAL COMPLIANT FEATURE ENGINEERING - TASK 1, STEP 4 
1. Transaction frequency and velocity
2. hour_of_day, day_of_week
3. time_since_signup
"""

import pandas as pd
import numpy as np
import sys
import os

# Add src path
sys.path.append('../src')

# Import your modular functions
from task1_data_analysis import engineer_time_features, transaction_velocity

# Load data
fraud_df = pd.read_csv('../data/raw/Fraud_Data.csv')
fraud_df['signup_time'] = pd.to_datetime(fraud_df['signup_time'])
fraud_df['purchase_time'] = pd.to_datetime(fraud_df['purchase_time'])

print("="*60)
print("TASK 1, STEP 4 - EXACT REQUIREMENTS")
print("="*60)

# -------------------------------------------------------------------
# 1. APPLY THE MODULAR FUNCTIONS (Covers basic requirements)
# -------------------------------------------------------------------
print("\n[1] Applying existing modular functions from src:")

# This gives us: hour_of_day, day_of_week, time_since_signup
fraud_df = engineer_time_features(fraud_df)
print("  ✓ engineer_time_features() added:")
print("    • hour_of_day")
print("    • day_of_week") 
print("    • time_since_signup (seconds)")

# This gives us: transaction_count
fraud_df = transaction_velocity(fraud_df)
print("\n  ✓ transaction_velocity() added:")
print("    • transaction_count (total per user)")

# -------------------------------------------------------------------
# 2. ENHANCE TO FULLY MEET "TRANSACTION FREQUENCY AND VELOCITY"
# -------------------------------------------------------------------
print("\n[2] Enhancing transaction frequency and velocity:")

# Sort by user and time
fraud_df = fraud_df.sort_values(['user_id', 'purchase_time'])

# Convert time_since_signup to days for velocity calculation
fraud_df['time_since_signup_days'] = fraud_df['time_since_signup'] / (3600 * 24)

# Transaction velocity: transactions per day
fraud_df['transaction_velocity'] = fraud_df['transaction_count'] / fraud_df['time_since_signup_days'].clip(lower=0.01)

print("  ✓ Added transaction velocity (transactions per day)")

# Time since previous transaction
fraud_df['time_since_previous_transaction'] = fraud_df.groupby('user_id')['purchase_time'].diff().dt.total_seconds() / 3600
fraud_df['time_since_previous_transaction'] = fraud_df['time_since_previous_transaction'].fillna(0)

print("  ✓ Added time since previous transaction (hours)")

# -------------------------------------------------------------------
# 3. SUMMARY OF REQUIREMENTS MET
# -------------------------------------------------------------------
print("\n" + "="*60)
print("REQUIREMENTS CHECKLIST")
print("="*60)

requirements = {
    "Transaction frequency and velocity": [
        "✓ transaction_count (frequency)",
        "✓ transaction_velocity (daily rate)", 
        "✓ time_since_previous_transaction (velocity metric)"
    ],
    "Time-based features": [
        "✓ hour_of_day (0-23)",
        "✓ day_of_week (0-6)"
    ],
    "time_since_signup": [
        "✓ time_since_signup (seconds)",
        "✓ time_since_signup_days (for velocity calculation)"
    ]
}

for req, features in requirements.items():
    print(f"\n{req}:")
    for feature in features:
        print(f"  {feature}")

# -------------------------------------------------------------------
# 4. SHOW FINAL FEATURES
# -------------------------------------------------------------------
print("\n" + "="*60)
print("FINAL ENGINEERED FEATURES")
print("="*60)

# Original columns
original_cols = ['user_id', 'signup_time', 'purchase_time', 'purchase_value',
                 'device_id', 'source', 'browser', 'sex', 'age',
                 'ip_address', 'class']

# New features created
new_features = [col for col in fraud_df.columns if col not in original_cols]

print(f"\nOriginal features: {len(original_cols)}")
print(f"New features created: {len(new_features)}")
print(f"Total features: {fraud_df.shape[1]}")

print("\nNew features (Task 1, Step 4):")
for i, feature in enumerate(new_features, 1):
    print(f"  {i:2}. {feature}")

# -------------------------------------------------------------------
# 5. SAVE FOR NEXT STEPS
# -------------------------------------------------------------------
# Save to processed folder
output_path = '../data/processed/fraud_data_step4_engineered.csv'
fraud_df.to_csv(output_path, index=False)

print("\n" + "="*60)
print(f"✓ Saved to: {output_path}")
print("✓ Ready for Step 5 (Data Transformation)")
print("="*60)

# Show sample
print("\nSample data with new features:")
sample = fraud_df[['user_id', 'hour_of_day', 'day_of_week', 
                   'transaction_count', 'transaction_velocity',
                   'time_since_signup_days', 'class']].head()
print(sample)

TASK 1, STEP 4 - EXACT REQUIREMENTS

[1] Applying existing modular functions from src:
  ✓ engineer_time_features() added:
    • hour_of_day
    • day_of_week
    • time_since_signup (seconds)

  ✓ transaction_velocity() added:
    • transaction_count (total per user)

[2] Enhancing transaction frequency and velocity:
  ✓ Added transaction velocity (transactions per day)
  ✓ Added time since previous transaction (hours)

REQUIREMENTS CHECKLIST

Transaction frequency and velocity:
  ✓ transaction_count (frequency)
  ✓ transaction_velocity (daily rate)
  ✓ time_since_previous_transaction (velocity metric)

Time-based features:
  ✓ hour_of_day (0-23)
  ✓ day_of_week (0-6)

time_since_signup:
  ✓ time_since_signup (seconds)
  ✓ time_since_signup_days (for velocity calculation)

FINAL ENGINEERED FEATURES

Original features: 11
New features created: 7
Total features: 18

New features (Task 1, Step 4):
   1. hour_of_day
   2. day_of_week
   3. time_since_signup
   4. transaction_count
   5. t

1.5: DATA TRANSFORMATION - A. FOR FRAUD DATASET

In [19]:
print("\nFRAUD DATA TRANSFORMATION")
print("-"*40)

# Load fraud data
fraud_df = pd.read_csv('../data/raw/Fraud_Data.csv')
print(f"Loaded: {fraud_df.shape}")

# Convert datetime
fraud_df['signup_time'] = pd.to_datetime(fraud_df['signup_time'])
fraud_df['purchase_time'] = pd.to_datetime(fraud_df['purchase_time'])

# Create basic features (from your src functions)
import sys
sys.path.append('../src')
try:
    from task1_data_analysis import engineer_time_features, transaction_velocity
    fraud_df = engineer_time_features(fraud_df)
    fraud_df = transaction_velocity(fraud_df)
    print("Applied src functions")
except:
    # If src not working, do basic features
    fraud_df['hour_of_day'] = fraud_df['purchase_time'].dt.hour
    fraud_df['day_of_week'] = fraud_df['purchase_time'].dt.dayofweek
    fraud_df['time_since_signup'] = (fraud_df['purchase_time'] - fraud_df['signup_time']).dt.total_seconds()
    fraud_df['transaction_count'] = fraud_df.groupby('user_id')['purchase_time'].transform('count')

print(f"\n1. Encoding categorical features...")

# One-Hot Encoding for source, browser, sex
fraud_encoded = pd.get_dummies(fraud_df, 
                               columns=['source', 'browser', 'sex'], 
                               drop_first=True)
print(f"   One-hot encoded: source, browser, sex")

print(f"\n2. Scaling numerical features...")

# Scale numerical columns (except ID columns and target)
numerical_cols = fraud_encoded.select_dtypes(include=[np.number]).columns.tolist()
exclude_cols = ['class', 'user_id', 'device_id', 'ip_address']
scale_cols = [col for col in numerical_cols if col not in exclude_cols]

scaler = StandardScaler()
fraud_encoded[scale_cols] = scaler.fit_transform(fraud_encoded[scale_cols])
print(f"   Scaled {len(scale_cols)} numerical features")

# Save
fraud_encoded.to_csv('../data/processed/fraud_transformed.csv', index=False)
print(f"\n✓ Saved: ../data/processed/fraud_transformed.csv ({fraud_encoded.shape})")


FRAUD DATA TRANSFORMATION
----------------------------------------
Loaded: (151112, 11)
Applied src functions

1. Encoding categorical features...
   One-hot encoded: source, browser, sex

2. Scaling numerical features...
   Scaled 6 numerical features

✓ Saved: ../data/processed/fraud_transformed.csv ((151112, 19))


PART B: CREDIT_CARD DATA TRANSFORMATION

In [20]:
print("\n\nCREDIT CARD DATA TRANSFORMATION")
print("-"*40)

# Load credit data
credit_df = pd.read_csv('../data/raw/creditcard.csv')
print(f"Loaded: {credit_df.shape}")

# For credit data, V1-V28 are already PCA scaled
# We only need to scale 'Time' and 'Amount'

print(f"\n1. Scaling 'Time' and 'Amount'...")

scaler = StandardScaler()
credit_df[['Time', 'Amount']] = scaler.fit_transform(credit_df[['Time', 'Amount']])
print(f"   Scaled: Time, Amount")

# Save
credit_df.to_csv('../data/processed/credit_transformed.csv', index=False)
print(f"\n✓ Saved: ../data/processed/credit_transformed.csv ({credit_df.shape})")



CREDIT CARD DATA TRANSFORMATION
----------------------------------------
Loaded: (284807, 31)

1. Scaling 'Time' and 'Amount'...
   Scaled: Time, Amount

✓ Saved: ../data/processed/credit_transformed.csv ((284807, 31))
